<a href="https://colab.research.google.com/github/AleksandarrP/AI-agent-working-with-ucs/blob/main/FinalExamAleksandarPetkovic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass, field
import numpy as np
from typing import Tuple, List, Dict, Optional, Set
import heapq
import time
import math
import pandas as pd

* Our project here is to make an agent with time complexity , moving hazard zones .
* Agent job is to rescue all prisoners .
* Agent will use UCS for shorthest way

In [ ]:
@dataclass(frozen=True)
class RescueState:

    x: int
    y: int
    carried: int  # -1 if we dont carry anyone
    rescued_mask: int
    t: int

In [ ]:
class TimeExpandedRescueGrid:


    def __init__(self, grid: List[str], start: Tuple[int, int],
                 safe: Tuple[int, int], workers: List[Tuple[int, int]],
                 deadline: int, blocked_schedule=None, hazard_schedule=None):
        self.grid = grid
        self.H = len(grid)
        self.W = len(grid[0])
        self.start = start
        self.safe = safe
        self.workers = workers
        self.deadline = deadline

        self.blocked_schedule = blocked_schedule or {}
        self.hazard_schedule = hazard_schedule or {}


        self.walls = set()
        for y, row in enumerate(grid):
            for x, ch in enumerate(row):
                if ch == '#':
                    self.walls.add((x, y))

    def initial_state(self) -> RescueState:

        return RescueState(self.start[0], self.start[1], -1, 0, 0)

    def in_bounds(self, x: int, y: int) -> bool:

        return 0 <= x < self.W and 0 <= y < self.H
        #checkhing if we are in grid

    def is_blocked(self, x: int, y: int, t: int) -> bool:

        if (x, y) in self.walls:
            return True
        return (t in self.blocked_schedule) and ((x, y) in self.blocked_schedule[t])
        #Checking if there if we are blocked by walls or temporary blocked (t)

    def is_hazard(self, x: int, y: int, t: int) -> bool:

        return (t in self.hazard_schedule) and ((x, y) in self.hazard_schedule[t])
        #Checking if the action is dangerous for us to take it

    def actions(self, s: RescueState) -> List[str]: #Making actions like in pdf

        if s.t >= self.deadline:
            return []
            #Checkhing if we crossed the deadline

        acts = []


        for a, (dx, dy) in [('U', (0,-1)), ('D', (0,1)), ('L', (-1,0)), ('R', (1,0))]:
            nx, ny = s.x + dx, s.y + dy
            nt = s.t + 1
            if self.in_bounds(nx, ny) and not self.is_blocked(nx, ny, nt):
                acts.append(a)

              #Moving Up down left and right if we are in grid and if we are not blocked


        if not self.is_blocked(s.x, s.y, s.t + 1):
            acts.append('W')
                  #We can stay at the same poisition if we are not blocked in the next interval


        if s.carried == -1:
            for i, (wx, wy) in enumerate(self.workers):
                if (s.x, s.y) == (wx, wy) and ((s.rescued_mask >> i) & 1) == 0:
                    acts.append(f'P{i}')
                    #Adding function so we can pick up the worker if we are at the same position as him and if we dont carry anyone else with 'P{i}' function


        if (s.x, s.y) == self.safe and s.carried != -1:
            acts.append('DROP')
            #and droping the worker at the safe place if we carry a worker and if its safe place

        return acts

    def result(self, s: RescueState, a: str) -> RescueState: #making a result function like it says in the instruction with state and action

        nt = s.t + 1 #We increase time
        x, y = s.x, s.y
        carried = s.carried
        rescued = s.rescued_mask

        if a in ['U','D','L','R']:
            dx, dy = {'U':(0,-1), 'D':(0,1), 'L':(-1,0), 'R':(1,0)}[a]   #Changing our positons with the moves we defined in actions
            x, y = x + dx, y + dy
        elif a == 'W':
            pass
        elif a.startswith('P'): #picking up worker
            i = int(a[1:])
            carried = i
        elif a == 'DROP':  #Droping our worker
            rescued = rescued | (1 << carried)
            carried = -1

        return RescueState(x, y, carried, rescued, nt)

    def step_cost(self, s: RescueState, a: str, ns: RescueState) -> int: #Making new function like in pdf with state action and new state

        base = 1


        if s.carried != -1 and a in ['U','D','L','R']:
            base = 2 #Here is base two because we carry a worker and as it says in pdf i put it to be more expensive which is i would say logical

        penalty = 0

        if self.is_hazard(ns.x, ns.y, ns.t):
            penalty = 5 #I added here penalty if we enter the hazard zone and set penalty to be 5

        return base + penalty

    def is_goal(self, s: RescueState) -> bool:

        all_rescued = s.rescued_mask == (1 << len(self.workers)) - 1
        return (all_rescued and s.carried == -1 and
                (s.x, s.y) == self.safe and s.t <= self.deadline)
 #Here i defined goal so all passengers are rescued and in safe place and we didnt pass deadline



In [ ]:
def ucs(env: TimeExpandedRescueGrid):

    start = env.initial_state()


    #List with cost ,current state,actions taken and states visited
    frontier = [(0,0, start, [], [start])]

    best_g = {start: 0}  # remebers the best cost found so far to reach each state
    explored = set()    # explored paths
    expanded = 0        # expanded nodes
    counter = 0        #
    t0 = time.perf_counter() #tine

    while frontier:
        g,_, state, actions, states = heapq.heappop(frontier)

        #Skip if we alredy expored this state
        if state in explored:
            continue

        #add state to explored once we explore it,and expand our node
        explored.add(state)
        expanded += 1

        #checkhing if we found goal
        if env.is_goal(state):
            return {
                'actions': actions,
                'states': states,
                'cost': g,
                'expanded': expanded,
                'runtime_s': time.perf_counter() - t0,
            }


        for a in env.actions(state): #loop over all valid actions from this state
            next_state = env.result(state, a) #Getting next sstate after aplying a
            next_g = g + env.step_cost(state, a, next_state) #getting cost

            if next_state not in explored and next_g < best_g.get(next_state, math.inf):#This ensures we dont expand again in the explored paths and we get the lowest cost
                best_g[next_state] = next_g
                counter +=1
                heapq.heappush(frontier, (next_g,counter, next_state, actions + [a], states + [next_state]))


    #And this is if no solutions where found
    return {'actions': None, 'states': None, 'cost': math.inf,
            'expanded': expanded, 'runtime_s': time.perf_counter() - t0}

In [ ]:
import random
def generate_random_grid(rows, cols, obstacle_prob):
    grid = []
    for r in range(rows):
        row = ""
        for c in range(cols):
            if random.random() < obstacle_prob:
                row += "#"
            else:
                row += "."
        grid.append(row)
    return grid
    #Here i just made a simple random grid with obstacle probability

In [ ]:
def make_test_instances():


    env1 = TimeExpandedRescueGrid(
        grid=generate_random_grid(10,10,0.8),
        start=(0, 0),
        safe=(9, 5),
        workers=[(3, 0), (6, 4)],
        deadline=30
    )



    env2 = TimeExpandedRescueGrid(
        grid=generate_random_grid(5,5,0.5),
        start=(0, 1),
        safe=(4, 4),
        workers=[(2, 0), (3, 1)],
        deadline=35
    )



    env3 = TimeExpandedRescueGrid(
        grid=generate_random_grid(7,7,0.2),
        start=(0, 0),
        safe=(4, 2),
        workers=[(0, 1), (1, 0)],
        deadline=40
    )

    return [
        ('Instance 1', env1),
        ('Instance 2', env2),
        ('Instance 3', env3),
    ] #Here i made 3 test enviorment

instances = make_test_instances()


In [ ]:
results = []
solutions = {}

for name, env in instances:
    out = ucs(env)
    solutions[name] = out

    success = out['actions'] is not None and math.isfinite(out['cost'])
    steps = len(out['actions']) if out['actions'] is not None else None

    results.append({
        "Instances": name,
        "Cost": out['cost'],
        "Steps": steps,
        "Expanded": out['expanded'],
        "time": round(out['runtime_s'], 4),
        'sucecess': "yes" if success else "No"
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

#aND FINAL RESULTS ARE HERE

 Instances  Cost  Steps  Expanded   time sucecess
Instance 1   inf    NaN         1 0.0000       No
Instance 2   inf    NaN       135 0.0013       No
Instance 3  30.0   20.0      3939 0.0577      yes
